# Maia Conversational Corpus Builder v3.0 — revisão consolidada

**Objetivo:** construir até **60.000 diálogos** (20.000 PT, 20.000 EN, 20.000 ES), divididos em conversação (7k/idioma), compreensão (4k), lógica (4k), instruções (3k) e incerteza (2k).

Esta versão substitui os dois notebooks anteriores. **Não falsifica aprovações de licença ou qualidade, não inventa dados para preencher cotas e nunca habilita SFT com corpus incompleto.** A aprovação de licença ocorre **por fonte e revisão**; a qualidade é controlada por filtros automáticos, auditoria de amostra por lote e registro de decisão. Os arquivos de revisão são gerados automaticamente para preenchimento pelo responsável. A primeira execução é diagnóstica.

**Atenção:** a meta de 60 mil depende da disponibilidade, licenças e diversidade das fontes; não é garantida por código. Os exemplos sintéticos abaixo são apenas testes e não entram no corpus de produção.

## 0. Dependências compatíveis com Colab
Não atualizar indiscriminadamente pandas/pyarrow/huggingface-hub. Reinicie a sessão se pip instalar algo.

In [ ]:
import importlib.util, subprocess, sys
missing=[p for p in ('datasets','huggingface_hub','pandas','tqdm') if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','datasets>=4,<5','huggingface-hub>=1.32,<3','tqdm>=4.66,<5'])
print('Pacotes ausentes inicialmente:',missing)


## 1. Configuração e arquivos de decisão
A configuração inicial produz amostras e relatórios. Para produção, altere `DRY_RUN=False` **depois** de revisar licenças e lotes. A política comercial é conservadora.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import os, json, csv, re, unicodedata, hashlib, random, itertools, datetime, math
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi
SEED=20261120
DRY_RUN=True
COMMERCIAL_ONLY=True
USE_DRIVE=True
MAX_PER_SOURCE=500 if DRY_RUN else 100000
MAX_OASST_NODES=2500 if DRY_RUN else 500000
TARGETS={'conversation':7000,'comprehension':4000,'logic':4000,'instruction':3000,'uncertainty':2000}
LANGS=('pt','en','es')
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v3_0') if USE_DRIVE else Path('/content/maia_corpus_v3_0')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT.mkdir(parents=True,exist_ok=True)
OUT=ROOT/('dry_run_revision' if DRY_RUN else 'full_revision');OUT.mkdir(parents=True,exist_ok=True)
DECISIONS=ROOT/'decisions';DECISIONS.mkdir(exist_ok=True)
SOURCE_DECISIONS=DECISIONS/'source_approvals.csv'
BATCH_DECISIONS=DECISIONS/'batch_reviews.csv'
SAMPLE_SIZE=25
print('Saída:',OUT,'| decisões:',DECISIONS)


## 2. Catálogo ampliado e aprovação de licenças

As fontes são **candidatas**. O catálogo não certifica licença nem disponibilidade. Cada aprovação deve referenciar a revisão exata do dataset, comprovar direitos de uso comercial e verificar proveniência/upstream. O notebook cria `source_approvals_template.csv` e lê `source_approvals.csv` se existir. Não aprova fontes por padrão. O esquema de cada fonte é validado durante a ingestão.

In [ ]:
SOURCES=[
 {'id':'oasst2','repo':'OpenAssistant/oasst2','kind':'oasst_tree','category':'conversation','split':'train','enabled':True},
 {'id':'ultrachatbr','repo':'recogna-nlp/UltrachatBR','kind':'chat','category':'conversation','split':'train','enabled':True},
 {'id':'aya','repo':'CohereLabs/aya_dataset','kind':'pair','category':'conversation','split':'train','enabled':True},
 {'id':'no_robots','repo':'HuggingFaceH4/no_robots','kind':'chat','category':'instruction','split':'train','enabled':False,'note':'CC BY-NC: excluded from commercial corpus'},
 {'id':'faquad','repo':'eraldoluis/faquad','kind':'qa','category':'comprehension','split':'train','enabled':False,'note':'Legacy dataset script; use verified local Parquet/JSON export only'},
 {'id':'assin2','repo':'ruanchaves/assin2','kind':'nli','category':'comprehension','split':'train','enabled':False,'note':'404: repository not found; verify alternative before enabling'},
 {'id':'canarim','repo':'dominguesm/Canarim-Instruct-PTBR-Dataset','kind':'pair','category':'instruction','split':'train','enabled':False,'note':'CC BY-NC: excluded from commercial corpus'},
 {'id':'squad_v2','repo':'rajpurkar/squad_v2','kind':'qa','category':'uncertainty','split':'train','enabled':False,'note':'CC BY-SA: provenance and obligations need review'},
]
# A verified local export can be supplied at ROOT/local_sources/faquad.parquet or .jsonl.
# Enabling it requires manual source approval, including the SHA-256 of the local file.
LOCAL_EXPORTS={'faquad':ROOT/'local_sources'/'faquad.parquet'}
ENABLE_VERIFIED_LOCAL_FAQUAD=False
if ENABLE_VERIFIED_LOCAL_FAQUAD and LOCAL_EXPORTS['faquad'].is_file():
    next(s for s in SOURCES if s['id']=='faquad')['enabled']=True
api=HfApi()
def read_csv(path,key='id'):
    if not path.exists():return {}
    with path.open(encoding='utf-8',newline='') as f:
        return {r[key]:r for r in csv.DictReader(f) if r.get(key)}
source_decisions=read_csv(SOURCE_DECISIONS)
manifest=[]
for s in SOURCES:
    x=dict(s)
    try:
        info=api.dataset_info(s['repo']);x['revision']=info.sha
        card=info.cardData
        x['card_license']=card.get('license') if isinstance(card,dict) else getattr(card,'license',None)
        x['metadata_status']='ok'
    except Exception as e:
        x.update(revision='',card_license='',metadata_status='error',metadata_error=str(e)[:300])
    if x['id']=='faquad' and ENABLE_VERIFIED_LOCAL_FAQUAD and LOCAL_EXPORTS['faquad'].is_file():
        x['revision']='local-sha256:'+hashlib.sha256(LOCAL_EXPORTS['faquad'].read_bytes()).hexdigest()
        x['metadata_status']='local_export'
        x['local_path']=str(LOCAL_EXPORTS['faquad'])
    d=source_decisions.get(x['id'],{})
    x['approved']=bool(x['revision'] and d.get('revision')==x['revision'] and
        d.get('decision','').strip().lower()=='approve' and
        all(d.get(k,'').strip() for k in ('reviewer','review_date','evidence_url')) and
        d.get('upstream_checked','').strip().lower()=='yes' and
        (not COMMERCIAL_ONLY or d.get('commercial_ok','').strip().lower()=='yes'))
    x['approval']=d
    x['approval_reason']=('approved' if x['approved'] else
       'not_enabled' if not x['enabled'] else
       'missing_or_stale_source_approval')
    manifest.append(x)
source_fields=['id','repo','revision','card_license','reviewer','review_date','evidence_url','upstream_checked','commercial_ok','decision','notes']
with (DECISIONS/'source_approvals_template.csv').open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=source_fields);w.writeheader()
    for x in manifest:
        row={k:x.get(k,'') for k in source_fields}
        if x['id'] in source_decisions:row.update({k:v for k,v in source_decisions[x['id']].items() if k in source_fields})
        row['revision']=x['revision']
        w.writerow(row)
(OUT/'sources_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
display(pd.DataFrame([{k:x.get(k) for k in ('id','repo','category','card_license','metadata_status','enabled','approved','approval_reason')} for x in manifest]))
print('Aprovações válidas:',sum(x['approved'] for x in manifest))
print('Edite e salve:',DECISIONS/'source_approvals.csv','(a partir do template, após auditoria real)')


## 3. Normalização, filtros e esquemas
Sem inferir idioma de forma silenciosa: só rótulos de idioma explícitos ou detecção de idioma com confiança suficiente; candidatos incertos são rejeitados. Registros importados preservam origem e revisão.

In [ ]:
try:
    from langdetect import detect_langs, DetectorFactory
    DetectorFactory.seed=SEED
except ImportError:
    detect_langs=None

def digest(s):return hashlib.sha256(str(s).encode('utf-8')).hexdigest()
def clean(s):return re.sub(r'\n{4,}','\n\n',unicodedata.normalize('NFC',str(s or '')).replace('\x00','').strip())
def role(x):
    x=str(x or '').lower();return 'user' if x in ('user','human','prompter') else 'assistant' if x in ('assistant','gpt','bot') else None
def lang_label(x):
    x=str(x or '').lower().replace('_','-')
    return next((l for l in LANGS if x==l or x.startswith(l+'-')),None)
def infer_lang(ms,raw):
    l=lang_label(raw)
    if l:return l
    if not detect_langs:return None
    sample=' '.join(m['content'] for m in ms if m['role']=='user')[:2000]
    if len(sample)<55:return None
    try:
        preds=detect_langs(sample)
        return preds[0].lang if preds and preds[0].lang in LANGS and preds[0].prob>=0.95 else None
    except Exception:return None
def validate(ms):
    if not 2<=len(ms)<=14:return 'turn_count'
    if ms[0]['role']!='user' or ms[-1]['role']!='assistant':return 'boundary_roles'
    for i,m in enumerate(ms):
        if m['role'] not in ('user','assistant') or (i and m['role']==ms[i-1]['role']):return 'role_alternation'
        if not 4<=len(m['content'])<=5000:return 'message_length'
        if 'http' in m['content'].lower() and len(m['content'])>4500:return 'possible_dump'
    if sum(len(m['content']) for m in ms)>18000:return 'conversation_too_long'
    return None
def make(src,raw_id,family,language,ms):
    if not isinstance(ms,list):return None,'schema_unrecognized'
    norm=[{'role':role(m.get('role')),'content':clean(m.get('content'))} for m in ms if isinstance(m,dict)]
    err=validate(norm)
    if err:return None,err
    lang=infer_lang(norm,language)
    if lang not in LANGS:return None,'language_unverified'
    canon=json.dumps(norm,ensure_ascii=False,sort_keys=True)
    return {'id':'maia_'+digest(src['id']+'|'+str(raw_id)+'|'+canon)[:24], 'category':src['category'],'lang':lang,'source_id':src['id'],'source_revision':src.get('revision'),'source_record_id':str(raw_id),'family_id':src['id']+'|'+str(family),'messages':norm,'content_sha256':digest(canon),'license_status':'approved' if src['approved'] else 'quarantine'},None
def messages_from_row(row):
    for k in ('messages','conversations','conversation','dialogue','chat','data'):
        val=row.get(k)
        if not isinstance(val,list):continue
        ms=[]
        for m in val:
            if not isinstance(m,dict):continue
            r=role(m.get('role',m.get('from',m.get('speaker'))))
            content=m.get('content',m.get('value',m.get('text')))
            if r and isinstance(content,str):ms.append({'role':r,'content':content})
        if ms:return ms
    return None
def pair_from_row(row):
    q=next((row[k] for k in ('inputs','prompt','instruction','question') if isinstance(row.get(k),str)),None)
    a=next((row[k] for k in ('targets','completion','output','answer','response') if isinstance(row.get(k),str)),None)
    return [{'role':'user','content':q},{'role':'assistant','content':a}] if q and a else None
def qa_from_row(row):
    context=next((row[k] for k in ('context','passage') if isinstance(row.get(k),str)),None)
    q=row.get('question');a=row.get('answers',row.get('answer'))
    if isinstance(a,dict):a=a.get('text')
    if isinstance(a,list):a=a[0] if a else None
    if not isinstance(q,str) or not isinstance(a,str) or not a.strip():return None
    return [{'role':'user','content':('Contexto:\n'+context+'\n\n' if context else '')+q},{'role':'assistant','content':a}]
def nli_from_row(row):
    p=row.get('sentence1',row.get('premise'));h=row.get('sentence2',row.get('hypothesis'));label=row.get('label')
    if not isinstance(p,str) or not isinstance(h,str) or label is None:return None
    # Os códigos de classe variam por dataset: não mapear rótulos sem auditar o schema.
    return None


## 4. Ingestão real, com limites e relatório de erros

OASST2 é reconstruído por `parent_id` e `message_tree_id`. Em `DRY_RUN`, truncar nós pode deixar árvores incompletas — por isso a cobertura diagnóstica não mede o potencial real da fonte. Erros de esquema e falta de idioma são contabilizados, não descartados silenciosamente.

In [ ]:
records=[];errors=[];stats=Counter();schema_examples={}
for src in manifest:
    if not src['enabled'] or src['metadata_status'] not in ('ok','local_export'):continue
    print('Ingestão:',src['id'])
    try:
        if src['metadata_status']=='local_export':
            local=Path(src['local_path'])
            if local.suffix=='.parquet':
                from datasets import Dataset
                ds=Dataset.from_parquet(str(local)).to_iterable_dataset()
            elif local.suffix=='.jsonl':
                ds=load_dataset('json',data_files=str(local),split='train',streaming=True)
            else:raise ValueError('Local export must be Parquet or JSONL')
        else:
            ds=load_dataset(src['repo'],split=src['split'],streaming=True,revision=src['revision'])
        limit=MAX_OASST_NODES if src['kind']=='oasst_tree' else MAX_PER_SOURCE
        rows=itertools.islice(ds,limit)
        if src['kind']=='oasst_tree':
            nodes={}
            for row in rows:
                if row.get('message_id') is not None:nodes[row['message_id']]=row
            candidates=[]
            for mid,row in nodes.items():
                if role(row.get('role'))!='assistant':continue
                chain=[];cur=row;seen=set()
                while cur is not None and cur.get('message_id') not in seen and len(chain)<14:
                    seen.add(cur['message_id']);r=role(cur.get('role'))
                    if not r:break
                    chain.append({'role':r,'content':cur.get('text','')});cur=nodes.get(cur.get('parent_id'))
                chain.reverse()
                candidates.append((mid,row.get('message_tree_id',mid),row.get('lang'),chain))
        else:
            candidates=[]
            for i,row in enumerate(rows):
                if i==0:schema_examples[src['id']]={'keys':list(row.keys()),'sample_types':{k:type(v).__name__ for k,v in row.items()}}
                ms=(messages_from_row(row) if src['kind']=='chat' else pair_from_row(row) if src['kind']=='pair' else qa_from_row(row) if src['kind']=='qa' else nli_from_row(row))
                candidates.append((row.get('id',i),row.get('id',i),row.get('lang',row.get('language')),ms))
        for rid,family,lang,ms in candidates:
            rec,err=make(src,rid,family,lang,ms)
            if err:stats[(src['id'],err)]+=1
            else:records.append(rec);stats[(src['id'],'normalized')]+=1
    except Exception as e:
        errors.append({'source':src['id'],'error':str(e)[:1200]});print('ERRO:',str(e)[:250])
(OUT/'ingest_errors.json').write_text(json.dumps(errors,ensure_ascii=False,indent=2),encoding='utf-8')
(OUT/'schema_examples.json').write_text(json.dumps(schema_examples,ensure_ascii=False,indent=2),encoding='utf-8')
pd.DataFrame([{'source':s,'reason':r,'count':v} for (s,r),v in stats.items()],columns=['source','reason','count']).to_csv(OUT/'ingest_stats.csv',index=False)
print('Normalizados:',len(records),'| erros de ingestão:',len(errors))
display(pd.DataFrame([{'source':s,'reason':r,'count':v} for (s,r),v in stats.items()]))


## 5. Deduplicação e diversidade

Deduplicação exata normalizada, além de limites de concentração por família. Não usar limiar cego de similaridade sem auditoria. Não se criam milhares de variações sintéticas para cumprir metas.

In [ ]:
def fingerprint(ms):
    return digest(re.sub(r'\s+',' ',unicodedata.normalize('NFKC',' '.join(m['role']+':'+m['content'] for m in ms)).casefold()).strip())
unique=[];rejected=[];seen=set();family_counts=Counter()
for r in sorted(records,key=lambda x:digest(str(SEED)+'|'+x['id'])):
    fp=fingerprint(r['messages'])
    if fp in seen:
        rejected.append({'id':r['id'],'reason':'duplicate_exact'});continue
    if family_counts[r['family_id']]>=5:
        rejected.append({'id':r['id'],'reason':'family_cap'});continue
    seen.add(fp);family_counts[r['family_id']]+=1;unique.append(r)
print('Únicos:',len(unique),'| rejeitados:',len(rejected))


## 6. Auditoria por lotes, não aprovação manual por registro

Um lote é `(source_id, revision, category, lang)`. O notebook seleciona uma **amostra reproduzível de 25 registros por lote**, exporta para revisão e aceita um arquivo `batch_reviews.csv` com a decisão do revisor. A aprovação **não é automática**: exige revisão da amostra e evidência, além dos filtros automáticos. Amostragem não garante ausência de problemas no lote inteiro; por isso mantemos rastreabilidade e relatórios de rejeição.

`batch_reviews.csv`: `batch_id,revision,reviewer,review_date,sample_checked,decision,notes` (decision = `approve` ou `reject`). A revisão é invalidada se a revisão da fonte mudar.

In [ ]:
def batch_id(r):return '|'.join([r['source_id'],str(r['source_revision']),r['category'],r['lang']])
batches=defaultdict(list)
for r in unique:batches[batch_id(r)].append(r)
review_rows=[];template=[];batch_signatures={}
for bid,items in sorted(batches.items()):
    # Batch signatures invalidate stale decisions if sample population changes.
    signature=digest('|'.join(sorted(r['content_sha256'] for r in items)))
    batch_signatures[bid]=signature
    rng=random.Random(int(digest(str(SEED)+bid)[:16],16))
    sample=rng.sample(items,min(SAMPLE_SIZE,len(items)))
    for r in sample:
        review_rows.append({'batch_id':bid,'batch_signature':signature,'id':r['id'],'source_id':r['source_id'],'lang':r['lang'],'category':r['category'],'messages':json.dumps(r['messages'],ensure_ascii=False)})
    template.append({'batch_id':bid,'revision':items[0]['source_revision'],'batch_signature':signature,'size':len(items),'sample_required':len(sample),'reviewer':'','review_date':'','sample_checked':'','decision':'','notes':''})
review_fields=['batch_id','batch_signature','id','source_id','lang','category','messages']
pd.DataFrame(review_rows,columns=review_fields).to_csv(OUT/'review_samples.csv',index=False)
batch_fields=['batch_id','revision','batch_signature','size','sample_required','reviewer','review_date','sample_checked','decision','notes']
if BATCH_DECISIONS.exists():
    batch_decisions=read_csv(BATCH_DECISIONS,'batch_id')
else:batch_decisions={}
with (DECISIONS/'batch_reviews_template.csv').open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=batch_fields);w.writeheader()
    for row in template:
        old=batch_decisions.get(row['batch_id'],{})
        if old.get('batch_signature')==row['batch_signature']:
            row.update({k:old[k] for k in ('reviewer','review_date','sample_checked','decision','notes') if k in old})
        w.writerow(row)
for r in unique:
    bid=batch_id(r);d=batch_decisions.get(bid,{})
    required=min(SAMPLE_SIZE,len(batches[bid]))
    valid=(d.get('revision')==str(r['source_revision']) and
           d.get('batch_signature')==batch_signatures[bid] and
           d.get('decision','').strip().lower()=='approve' and
           d.get('reviewer','').strip() and d.get('review_date','').strip() and
           d.get('sample_checked','').isdigit() and int(d['sample_checked'])>=required)
    r['quality_status']='batch_approved' if valid else 'pending_batch_review'
    r['batch_id']=bid
approved=[r for r in unique if r['license_status']=='approved' and r['quality_status']=='batch_approved']
quarantine=[r for r in unique if r not in approved]
print('Aprovados:',len(approved),'| quarentena:',len(quarantine),'| lotes:',len(batches))
print('Revise as amostras:',OUT/'review_samples.csv')
print('Após revisão, salve as decisões em:',BATCH_DECISIONS)


## 7. Cotas e splits por família

A família é atribuída a um único split; nenhuma variante de uma árvore pode aparecer em treino e teste. As cotas de 20 mil por idioma são metas **totais** de cada categoria, somando os três splits. A distribuição 90/5/5 é aproximada e determinística.

In [ ]:
def assign_split(family):
    u=int(digest(str(SEED)+'|'+family)[:12],16)/float(16**12)
    return 'train' if u<.90 else 'validation' if u<.95 else 'test'
selected=[];counts=Counter()
for r in sorted(approved,key=lambda x:digest(str(SEED)+'|'+x['id'])):
    key=(r['category'],r['lang'])
    if counts[key]<TARGETS.get(r['category'],0):
        selected.append(dict(r,split=assign_split(r['family_id'])));counts[key]+=1
coverage=[{'category':c,'lang':l,'target':t,'obtained':counts[c,l],'missing':t-counts[c,l]} for c,t in TARGETS.items() for l in LANGS]
display(pd.DataFrame(coverage))


## 8. Exportação e travamento do SFT

`READY_FOR_SFT.json` só aparece quando as 15 cotas estiverem completas, os três splits tiverem exemplos, as licenças e lotes estiverem aprovados, não houver erros de ingestão e `DRY_RUN=False`. Os arquivos vazios são **intencionais** enquanto não houver fontes aprovadas. O relatório explica os bloqueios.

In [ ]:
def write_jsonl(path,items):
    with path.open('w',encoding='utf-8') as f:
        for item in items:f.write(json.dumps(item,ensure_ascii=False)+'\n')
for split in ('train','validation','test'):write_jsonl(OUT/(split+'.jsonl'),(r for r in selected if r['split']==split))
write_jsonl(OUT/'quarantine.jsonl',quarantine)
write_jsonl(OUT/'rejected.jsonl',rejected)
pd.DataFrame(coverage).to_csv(OUT/'coverage.csv',index=False)
missing=sum(x['missing'] for x in coverage)
source_counts=Counter(r['source_id'] for r in selected)
split_counts=Counter(r['split'] for r in selected)
blockers=[]
if DRY_RUN:blockers.append('dry_run_enabled')
if missing:blockers.append('coverage_incomplete')
if errors:blockers.append('ingest_errors')
if any(split_counts[s]==0 for s in ('train','validation','test')):blockers.append('split_empty')
if any(r['license_status']!='approved' or r['quality_status']!='batch_approved' for r in selected):blockers.append('unapproved_selected')
if len({fingerprint(r['messages']) for r in selected})!=len(selected):blockers.append('duplicate_selected')
ready=not blockers
report={'seed':SEED,'dry_run':DRY_RUN,'commercial_only':COMMERCIAL_ONLY,'target_total':sum(TARGETS.values())*len(LANGS),'selected_total':len(selected),'quarantine_total':len(quarantine),'rejected_total':len(rejected),'missing_total':missing,'source_counts':dict(source_counts),'split_counts':dict(split_counts),'ingest_errors':len(errors),'approved_sources':[s['id'] for s in manifest if s['approved']],'blockers':blockers,'status':'READY_FOR_SFT' if ready else 'INCOMPLETE_DO_NOT_TRAIN','created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
(OUT/'build_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
marker=OUT/'READY_FOR_SFT.json'
if ready:marker.write_text(json.dumps({'status':'READY_FOR_SFT','report_sha256':hashlib.sha256((OUT/'build_report.json').read_bytes()).hexdigest()},indent=2),encoding='utf-8')
else:marker.unlink(missing_ok=True)
checks=[{'file':p.name,'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(OUT.iterdir()) if p.is_file() and p.name!='checksums.csv']
pd.DataFrame(checks).to_csv(OUT/'checksums.csv',index=False)
print(json.dumps(report,ensure_ascii=False,indent=2))
if not ready:print('ATENÇÃO: corpus incompleto. NÃO iniciar SFT.')


## 9. Procedimento de produção — corrigido

1. Execute inicialmente `DRY_RUN=True`. OASST2, UltraChatBR e Aya permanecem candidatos; **nenhuma fonte é aprovada automaticamente** por uma licença declarada no cartão.
2. Examine `sources_manifest.json`, a licença **do dataset e dos dados upstream**, e `schema_examples.json`. Documente a auditoria em `decisions/source_approvals.csv`, copiando `source_approvals_template.csv` e preenchendo `reviewer`, `review_date`, `evidence_url`, `upstream_checked=yes`, `commercial_ok=yes` e `decision=approve` somente após verificação. Não invente essas informações.
3. Revise `review_samples.csv`. Copie `batch_reviews_template.csv` para `decisions/batch_reviews.csv` e registre decisões reais. Uma decisão é invalidada se a revisão do dataset **ou o conteúdo do lote** mudar. **Lotes de dry-run não podem ser reutilizados automaticamente no full build.**
4. Reexecute o notebook para aplicar as decisões. Depois, configure `DRY_RUN=False`, revise os novos lotes e reexecute novamente.
5. FaQuAD foi desabilitado por depender de um script legado. Para usá-lo, obtenha e audite um export **autêntico** Parquet/JSONL, coloque-o em `ROOT/local_sources/faquad.parquet` e defina `ENABLE_VERIFIED_LOCAL_FAQUAD=True`. A aprovação exige o SHA-256 do arquivo, registrado como revisão. Não foi criado um export fictício.
6. ASSIN2 foi desabilitado por 404; No Robots e Canarim foram desabilitados por licença não comercial declarada. SQuAD v2 permanece desabilitado aguardando análise de atribuição e compartilhamento.
7. A meta de 60.000 exemplos não está garantida por estas fontes. Faltam fontes de lógica, instruções e incerteza auditadas, além de validação semântica e da distribuição por idioma. O SFT permanece bloqueado até que a cobertura e as aprovações estejam completas.

**Nota:** `source_approvals_template.csv` e `batch_reviews_template.csv` são modelos, não decisões aprovadas. Não altere artificialmente os flags para obter 60.000 exemplos.
